# WiDS 2026 — Pseudo-Observations + Hard Distance Rules (v5)

**Why this notebook:** Current score 0.96837. Top scorers > 0.99.  
The gap comes from two problems our survival models don't fully solve:
1. **Far fires** (dist > 5km): models predict 0.05–0.20 when the true answer is ~0.00 → direct Brier penalty
2. **Close fires** (dist < 5km): all 69 hits are here; the Brier loss is entirely about *timing* precision

**Two targeted fixes:**
- **Pseudo-observation XGBoost** — directly minimises Brier (MSE on jackknife pseudo-labels)
- **Distance-stratified hard rules** — enforces near-zero probabilities for fires the model cannot possibly misclassify

**Final pipeline:** Pseudo-obs XGBoost + GBT survival ensemble, distance-gated output

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import optuna
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from lifelines import KaplanMeierFitter, CoxPHFitter
from sksurv.ensemble import GradientBoostingSurvivalAnalysis, RandomSurvivalForest
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.float_format', '{:.5f}'.format)

SEED          = 42
DATA_DIR      = 'Data/'
HORIZONS      = [12, 24, 48, 72]
BRIER_WEIGHTS = {24: 0.3, 48: 0.4, 72: 0.3}

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  CONFIG  ← all decisions here
# ─────────────────────────────────────────────────────────────────

# Hard distance rules
# Fires within CLOSE_DIST_M will have predictions scaled UP toward CLOSE_FLOOR
# Fires beyond FAR_DIST_M will be capped at FAR_CAP (essentially ~0)
CLOSE_DIST_M  = 5_000      # metres — all training hits are within this
FAR_DIST_M    = 15_000     # metres — no training hits beyond this; be conservative
FAR_CAP       = 0.05       # max probability assigned to any far fire
CLOSE_FLOOR   = 0.70       # min prob_72h for a confirmed-close fire

# Pseudo-observation XGBoost
PSEUDO_XGB_PARAMS = {
    'objective':        'reg:squarederror',
    'max_depth':         3,
    'learning_rate':     0.05,
    'n_estimators':      500,
    'subsample':         0.75,
    'colsample_bytree':  0.75,
    'min_child_weight':  5,
    'reg_alpha':         0.1,
    'reg_lambda':        1.0,
    'random_state':      SEED,
}
PSEUDO_OPTUNA_TRIALS = 60   # set 0 to skip Optuna and use PSEUDO_XGB_PARAMS

# Survival model (GBT from notebook 04 — paste best params here)
GBT_PARAMS = {
    'n_estimators':     200,
    'learning_rate':    0.05,
    'max_depth':         3,
    'subsample':         0.8,
    'min_samples_leaf':  12,
    'random_state':      SEED,
}

# Ensemble weight: pseudo-obs XGBoost vs GBT survival
# Adjust based on CV scores from Section 5
PSEUDO_WEIGHT   = 0.65
SURVIVAL_WEIGHT = 0.35

N_CV = 5
print('Config loaded.')

## 1. Data & Feature Engineering

In [ ]:
train_raw = pd.read_csv(DATA_DIR + 'train.csv')
test_raw  = pd.read_csv(DATA_DIR + 'test.csv')

DROP_REDUNDANT = [
    'relative_growth_0_5h', 'area_growth_rate_ha_per_h',
    'centroid_displacement_m', 'centroid_speed_m_per_h',
    'closing_speed_m_per_h', 'projected_advance_m',
    'dist_slope_ci_0_5h', 'closing_speed_abs_m_per_h',
]
DROP_NOISE = [
    'event_start_hour', 'event_start_dayofweek', 'event_start_month',
    'along_track_speed', 'cross_track_component', 'dist_accel_m_per_h2',
]

def build_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['log1p_dist_min']    = np.log1p(df['dist_min_ci_0_5h'])
    df['dist_under_5km']    = (df['dist_min_ci_0_5h'] < 5_000).astype(int)
    df['is_growing']        = (df['area_growth_abs_0_5h'] > 0).astype(int)
    df['speed_x_alignment'] = df['radial_growth_rate_m_per_h'] * df['alignment_abs']
    # Restore closing_speed_abs as it carries signal for timing of close fires
    # (it was dropped for redundancy with dist_std_ci, but r=0.997 was in the
    #  full feature set — here we keep ONLY closing_speed_abs, not dist_std)
    drop = [c for c in DROP_REDUNDANT + DROP_NOISE + ['dist_std_ci_0_5h'] if c in df.columns]
    df   = df.drop(columns=drop)
    meta = [c for c in ['event_id', 'time_to_hit_hours', 'event'] if c in df.columns]
    return df.drop(columns=meta)


X_train   = build_features(train_raw)
X_test    = build_features(test_raw)
y_event   = train_raw['event'].values.astype(bool)
y_time    = train_raw['time_to_hit_hours'].values
dist_train = train_raw['dist_min_ci_0_5h'].values
dist_test  = test_raw['dist_min_ci_0_5h'].values

print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')

# Confirm the distance threshold rule holds in training data
close_mask = dist_train < CLOSE_DIST_M
far_mask   = dist_train >= CLOSE_DIST_M
print(f'\nDistance threshold sanity check (training set):')
print(f'  dist < {CLOSE_DIST_M/1000:.0f}km : n={close_mask.sum()}, '
      f'event rate = {y_event[close_mask].mean():.2%}')
print(f'  dist >= {CLOSE_DIST_M/1000:.0f}km: n={far_mask.sum()}, '
      f'event rate = {y_event[far_mask].mean():.2%}')
print(f'\nTest set distance distribution:')
print(f'  dist < 5km  : {(dist_test < 5_000).sum()} fires')
print(f'  5km–15km    : {((dist_test >= 5_000) & (dist_test < 15_000)).sum()} fires')
print(f'  15km–50km   : {((dist_test >= 15_000) & (dist_test < 50_000)).sum()} fires')
print(f'  > 50km      : {(dist_test >= 50_000).sum()} fires')

## 2. Metrics & Utilities

In [ ]:
def brier_at_horizon(ye, yt, pred, H):
    mask  = ~((ye == 0) & (yt < H))
    y_obs = ((ye == 1) & (yt <= H)).astype(float)
    return float(np.mean((y_obs[mask] - pred[mask]) ** 2))

def weighted_brier(ye, yt, pd_):
    return sum(w * brier_at_horizon(ye, yt, pd_[H], H) for H, w in BRIER_WEIGHTS.items())

def cindex(ye, yt, risk):
    return concordance_index_censored(ye.astype(bool), yt, risk)[0]

def hybrid(ci, wb):
    return 0.3 * ci + 0.7 * (1.0 - wb)

def evaluate(ye, yt, pd_, label=''):
    ci = cindex(ye, yt, pd_[24])
    wb = weighted_brier(ye, yt, pd_)
    return {'Model': label, 'C-index': ci,
            **{f'B@{H}h': brier_at_horizon(ye, yt, pd_[H], H) for H in HORIZONS},
            'W-Brier': wb, 'Hybrid': hybrid(ci, wb)}

def enforce_monotonicity(pd_):
    mat = np.column_stack([pd_[H] for H in HORIZONS]).clip(0, 1)
    mat = np.maximum.accumulate(mat, axis=1)
    return {H: mat[:, i] for i, H in enumerate(HORIZONS)}

print('Metrics defined.')

## 3. Diagnosing Where Current Models Lose Brier Points

In [ ]:
# Quick GBT on full train — examine where predictions are wrong
_m = GradientBoostingSurvivalAnalysis(**GBT_PARAMS)
_m.fit(X_train.values, Surv.from_arrays(y_event, y_time))

def _safe_eval(fn, t):
    return float(fn(float(np.clip(t, fn.x[0], fn.x[-1]))))

fns   = _m.predict_survival_function(X_train.values)
S_mat = np.array([[_safe_eval(f, t) for t in HORIZONS] for f in fns])
gbt_train_probs = {H: 1.0 - S_mat[:, i] for i, H in enumerate(HORIZONS)}

print('GBT train predictions — mean prob by distance zone:')
print(f'{"Zone":<25} {"n":>5} {"P12":>8} {"P24":>8} {"P48":>8} {"P72":>8}')
print('-' * 62)
zones = [
    ('< 5km (all hits)',     dist_train < 5_000),
    ('5–15km',               (dist_train >= 5_000)  & (dist_train < 15_000)),
    ('15–50km',              (dist_train >= 15_000) & (dist_train < 50_000)),
    ('50–200km',             (dist_train >= 50_000) & (dist_train < 200_000)),
    ('> 200km',              dist_train >= 200_000),
]
for label, mask in zones:
    if mask.sum() == 0:
        continue
    means = [gbt_train_probs[H][mask].mean() for H in HORIZONS]
    print(f'{label:<25} {mask.sum():>5}  {means[0]:>7.4f}  {means[1]:>7.4f}  {means[2]:>7.4f}  {means[3]:>7.4f}')

print()
print('Brier contribution by zone (horizon=72h):')
for label, mask in zones:
    if mask.sum() == 0:
        continue
    # For 72h: y_obs = 1 if hit by 72h, 0 if censored at any time
    y_obs = ((y_event == 1) & (y_time <= 72)).astype(float)
    b = np.mean((y_obs[mask] - gbt_train_probs[72][mask])**2)
    print(f'  {label:<25} Brier@72h={b:.5f}  (n={mask.sum()})')

## 4. Jackknife Pseudo-Observations

For each fire `i` and horizon `H`, the pseudo-observation is:

`ψᵢ(H) = n × F̂ₙ(H) − (n−1) × F̂ₙ₋ᵢ(H)`

where `F̂(H) = 1 − Ŝ(H)` is the KM estimator of P(hit by H).

**Key property:** `E[ψᵢ(H) | Xᵢ] = P(hit by H | Xᵢ)` — censoring is handled automatically. Training XGBoost to predict ψᵢ(H) directly minimises Brier score.

In [ ]:
def compute_pseudo_observations(y_event: np.ndarray,
                                 y_time:  np.ndarray,
                                 horizons: list) -> dict:
    """
    Jackknife pseudo-observations for P(hit by H) at each horizon.
    Returns {H: array of shape (n,)} — one pseudo-obs per fire.
    """
    n    = len(y_event)
    pseudo = {H: np.zeros(n) for H in horizons}

    # Full-sample KM estimates
    kmf_full = KaplanMeierFitter()
    kmf_full.fit(y_time, event_observed=y_event)
    F_full = {H: 1.0 - kmf_full.survival_function_at_times(H).values[0]
              for H in horizons}

    # Leave-one-out KM estimates
    for i in range(n):
        mask  = np.ones(n, dtype=bool)
        mask[i] = False
        kmf_i = KaplanMeierFitter()
        kmf_i.fit(y_time[mask], event_observed=y_event[mask])

        for H in horizons:
            F_i          = 1.0 - kmf_i.survival_function_at_times(H).values[0]
            pseudo[H][i] = n * F_full[H] - (n - 1) * F_i

    # Clip pseudo-observations to [−0.5, 1.5] before passing to XGBoost
    # (extreme pseudo-obs from very influential points can destabilise training)
    for H in horizons:
        pseudo[H] = pseudo[H].clip(-0.5, 1.5)

    return pseudo


print('Computing jackknife pseudo-observations (221 leave-one-out KM fits × 4 horizons)...')
pseudo_obs = compute_pseudo_observations(y_event, y_time, HORIZONS)
print('Done.')

print('\nPseudo-observation summary per horizon:')
for H in HORIZONS:
    ps = pseudo_obs[H]
    print(f'  H={H:>2}h: mean={ps.mean():.4f}  std={ps.std():.4f}  '
          f'min={ps.min():.4f}  max={ps.max():.4f}  '
          f'in[0,1]: {((ps>=0)&(ps<=1)).mean():.1%}')

In [ ]:
# ── How do pseudo-observations relate to the true outcome? ──
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, H in zip(axes, HORIZONS):
    y_obs = ((y_event == 1) & (y_time <= H)).astype(int)
    # Show pseudo-obs distributions for hit vs not-hit-by-H fires
    ax.hist(pseudo_obs[H][y_obs == 1], bins=20, alpha=0.7,
            color='#e74c3c', label=f'Hit by {H}h', density=True)
    ax.hist(pseudo_obs[H][y_obs == 0], bins=20, alpha=0.5,
            color='#3498db', label=f'Not hit by {H}h', density=True)
    ax.axvline(0.5, color='black', ls='--', lw=1)
    ax.set_title(f'Pseudo-obs at {H}h', fontsize=11)
    ax.set_xlabel('ψ value')
    ax.legend(fontsize=8)

plt.suptitle('Pseudo-observation distributions: hit vs not-hit fires',
             fontsize=12)
plt.tight_layout()
plt.show()

## 5. XGBoost on Pseudo-Observations

In [ ]:
def train_pseudo_xgb(X: pd.DataFrame, pseudo: dict, params: dict) -> dict:
    """Train one XGBoost model per horizon on pseudo-observations.
    Returns {H: fitted XGBRegressor}.
    """
    models = {}
    for H in HORIZONS:
        model = xgb.XGBRegressor(**params)
        model.fit(X.values, pseudo[H])
        models[H] = model
    return models


def predict_pseudo_xgb(models: dict, X: pd.DataFrame) -> dict:
    """Predict P(hit by H) for each horizon. Clips to [0,1]."""
    return {H: models[H].predict(X.values).clip(0, 1) for H in HORIZONS}


def cv_pseudo_xgb(X, ye, yt, pseudo, params, n_splits=N_CV):
    """5-fold CV for the pseudo-observation XGBoost approach."""
    skf  = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    rows = []
    for fold, (tr, val) in enumerate(skf.split(X, ye), 1):
        # Re-compute pseudo-obs on the training fold only
        fold_pseudo = compute_pseudo_observations(ye[tr], yt[tr], HORIZONS)
        models      = train_pseudo_xgb(X.iloc[tr], fold_pseudo, params)
        pd_         = predict_pseudo_xgb(models, X.iloc[val])
        pd_         = enforce_monotonicity(pd_)
        rows.append(evaluate(ye[val], yt[val], pd_, f'PseudoXGB fold-{fold}'))
    df   = pd.DataFrame(rows)
    mean = df[['C-index', 'W-Brier', 'Hybrid']].mean()
    std  = df[['C-index', 'W-Brier', 'Hybrid']].std()
    return mean.to_dict(), std.to_dict(), df


print('Pseudo-XGB helpers defined.')

In [ ]:
# ── Optuna tuning for pseudo-obs XGBoost ──
TUNED_PSEUDO_PARAMS = dict(PSEUDO_XGB_PARAMS)  # fallback to defaults

if PSEUDO_OPTUNA_TRIALS > 0:
    print(f'Tuning pseudo-obs XGBoost ({PSEUDO_OPTUNA_TRIALS} trials)...')

    def pseudo_objective(trial):
        params = {
            'objective':       'reg:squarederror',
            'max_depth':        trial.suggest_int('max_depth', 2, 5),
            'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'n_estimators':     trial.suggest_int('n_estimators', 100, 600),
            'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 3, 20),
            'reg_alpha':        trial.suggest_float('reg_alpha', 1e-3, 1.0, log=True),
            'reg_lambda':       trial.suggest_float('reg_lambda', 0.5, 5.0),
            'random_state':     SEED,
        }
        try:
            mean, _, _ = cv_pseudo_xgb(X_train, y_event, y_time, pseudo_obs, params, n_splits=5)
            return mean['Hybrid']
        except Exception:
            return 0.0

    study = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(pseudo_objective, n_trials=PSEUDO_OPTUNA_TRIALS,
                   show_progress_bar=True)

    TUNED_PSEUDO_PARAMS = study.best_params.copy()
    TUNED_PSEUDO_PARAMS['objective']    = 'reg:squarederror'
    TUNED_PSEUDO_PARAMS['random_state'] = SEED
    print(f'Best CV Hybrid: {study.best_value:.5f}')
    print(f'Best params: {TUNED_PSEUDO_PARAMS}')
else:
    print('Using default pseudo-XGB params (set PSEUDO_OPTUNA_TRIALS > 0 to tune).')

In [ ]:
# ── CV evaluation with tuned params ──
print('Running CV for pseudo-obs XGBoost (tuned params)...')
pseudo_mean, pseudo_std, pseudo_cv_df = cv_pseudo_xgb(
    X_train, y_event, y_time, pseudo_obs, TUNED_PSEUDO_PARAMS)

print(f"\nPseudo-obs XGBoost CV:")
print(f"  Hybrid  = {pseudo_mean['Hybrid']:.5f} ± {pseudo_std['Hybrid']:.5f}")
print(f"  C-index = {pseudo_mean['C-index']:.5f} ± {pseudo_std['C-index']:.5f}")
print(f"  W-Brier = {pseudo_mean['W-Brier']:.5f} ± {pseudo_std['W-Brier']:.5f}")

## 6. GBT Survival Model (for C-index diversity)

In [ ]:
def cv_gbt_survival(X, ye, yt, params, n_splits=N_CV):
    skf  = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    rows = []
    for fold, (tr, val) in enumerate(skf.split(X, ye), 1):
        m   = GradientBoostingSurvivalAnalysis(**params)
        m.fit(X.iloc[tr].values, Surv.from_arrays(ye[tr], yt[tr]))
        fns = m.predict_survival_function(X.iloc[val].values)
        S   = np.array([[_safe_eval(f, t) for t in HORIZONS] for f in fns])
        pd_ = enforce_monotonicity({H: 1.0 - S[:, i] for i, H in enumerate(HORIZONS)})
        rows.append(evaluate(ye[val], yt[val], pd_, f'GBT fold-{fold}'))
    df   = pd.DataFrame(rows)
    mean = df[['C-index', 'W-Brier', 'Hybrid']].mean()
    return mean.to_dict(), df


def _safe_eval(fn, t):
    return float(fn(float(np.clip(t, fn.x[0], fn.x[-1]))))


print('Running GBT survival CV...')
gbt_mean, gbt_cv_df = cv_gbt_survival(X_train, y_event, y_time, GBT_PARAMS)

print(f"GBT Survival CV:")
print(f"  Hybrid  = {gbt_mean['Hybrid']:.5f}")
print(f"  C-index = {gbt_mean['C-index']:.5f}")
print(f"  W-Brier = {gbt_mean['W-Brier']:.5f}")

## 7. Hard Distance Rules

Training data is unambiguous:
- Every hit fire had dist < 5km
- No censored fire had dist < 5km

We apply two post-processing rules:
1. **Far fires** (dist > `FAR_DIST_M`): cap all probabilities at `FAR_CAP` — the model cannot assign > 5% probability to a fire that has never hit in training within this distance range
2. **Close fires** (dist < `CLOSE_DIST_M`): enforce a minimum `prob_72h >= CLOSE_FLOOR` — these fires should be treated as nearly certain to hit

`FAR_DIST_M = 15km` is conservative (not 5km) to allow for test fires that might behave differently from training.

In [ ]:
def apply_distance_rules(prob_dict: dict, dist_m: np.ndarray) -> dict:
    """
    Apply hard rules based on distance to nearest evacuation zone.

    Far fires  (dist > FAR_DIST_M):   cap all horizon probabilities at FAR_CAP
    Close fires (dist < CLOSE_DIST_M): enforce prob_72h >= CLOSE_FLOOR
    """
    pd_ = {H: prob_dict[H].copy() for H in HORIZONS}

    far_mask   = dist_m > FAR_DIST_M
    close_mask = dist_m < CLOSE_DIST_M

    # Cap far-fire probabilities
    for H in HORIZONS:
        pd_[H][far_mask] = np.minimum(pd_[H][far_mask], FAR_CAP)

    # Raise close-fire prob_72h floor
    pd_[72][close_mask] = np.maximum(pd_[72][close_mask], CLOSE_FLOOR)

    return enforce_monotonicity(pd_)


# ── Show impact on training data (in-sample) ──
gbt_full = GradientBoostingSurvivalAnalysis(**GBT_PARAMS)
gbt_full.fit(X_train.values, Surv.from_arrays(y_event, y_time))
fns_tr = gbt_full.predict_survival_function(X_train.values)
S_tr   = np.array([[_safe_eval(f, t) for t in HORIZONS] for f in fns_tr])
raw_probs  = enforce_monotonicity({H: 1.0-S_tr[:,i] for i,H in enumerate(HORIZONS)})
ruled_probs = apply_distance_rules(raw_probs, dist_train)

row_raw   = evaluate(y_event, y_time, raw_probs,   'GBT raw (in-sample)')
row_ruled = evaluate(y_event, y_time, ruled_probs, 'GBT + distance rules (in-sample)')

print('Impact of distance rules on GBT in-sample scores:')
print(f'  Without rules: Hybrid={row_raw["Hybrid"]:.5f}  '
      f'W-Brier={row_raw["W-Brier"]:.5f}  C-index={row_raw["C-index"]:.5f}')
print(f'  With rules:    Hybrid={row_ruled["Hybrid"]:.5f}  '
      f'W-Brier={row_ruled["W-Brier"]:.5f}  C-index={row_ruled["C-index"]:.5f}')
print('(In-sample is optimistic — main benefit shows up on held-out data)')

In [ ]:
# ── CV evaluation of GBT + distance rules ──
def cv_gbt_with_rules(X, ye, yt, dist_arr, params, n_splits=N_CV):
    skf  = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    rows = []
    for fold, (tr, val) in enumerate(skf.split(X, ye), 1):
        m   = GradientBoostingSurvivalAnalysis(**params)
        m.fit(X.iloc[tr].values, Surv.from_arrays(ye[tr], yt[tr]))
        fns = m.predict_survival_function(X.iloc[val].values)
        S   = np.array([[_safe_eval(f, t) for t in HORIZONS] for f in fns])
        pd_ = enforce_monotonicity({H: 1.0 - S[:, i] for i, H in enumerate(HORIZONS)})
        pd_ = apply_distance_rules(pd_, dist_arr[val])
        rows.append(evaluate(ye[val], yt[val], pd_, f'GBT+rules fold-{fold}'))
    df   = pd.DataFrame(rows)
    mean = df[['C-index', 'W-Brier', 'Hybrid']].mean()
    return mean.to_dict(), df


print('Running GBT + distance rules CV...')
gbt_ruled_mean, gbt_ruled_df = cv_gbt_with_rules(
    X_train, y_event, y_time, dist_train, GBT_PARAMS)

print(f"GBT + distance rules CV:")
print(f"  Hybrid  = {gbt_ruled_mean['Hybrid']:.5f}  "
      f"(was {gbt_mean['Hybrid']:.5f})")
print(f"  C-index = {gbt_ruled_mean['C-index']:.5f}")
print(f"  W-Brier = {gbt_ruled_mean['W-Brier']:.5f}  "
      f"(was {gbt_mean['W-Brier']:.5f})")

## 8. Full Pipeline: Pseudo-obs + GBT + Distance Rules

In [ ]:
def cv_full_pipeline(X, ye, yt, dist_arr, pseudo_params, gbt_params,
                     w_pseudo, w_gbt, n_splits=N_CV):
    """
    Full pipeline CV:
    1. Pseudo-obs XGBoost
    2. GBT survival
    3. Weighted ensemble
    4. Distance rules
    """
    skf  = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    rows = []

    for fold, (tr, val) in enumerate(skf.split(X, ye), 1):
        # -- Pseudo-obs model --
        fold_pseudo    = compute_pseudo_observations(ye[tr], yt[tr], HORIZONS)
        pseudo_models  = train_pseudo_xgb(X.iloc[tr], fold_pseudo, pseudo_params)
        pseudo_preds   = predict_pseudo_xgb(pseudo_models, X.iloc[val])

        # -- GBT survival --
        m_gbt  = GradientBoostingSurvivalAnalysis(**gbt_params)
        m_gbt.fit(X.iloc[tr].values, Surv.from_arrays(ye[tr], yt[tr]))
        fns    = m_gbt.predict_survival_function(X.iloc[val].values)
        S      = np.array([[_safe_eval(f, t) for t in HORIZONS] for f in fns])
        gbt_preds = {H: 1.0 - S[:, i] for i, H in enumerate(HORIZONS)}

        # -- Weighted ensemble --
        total = w_pseudo + w_gbt
        ens   = {H: (w_pseudo * pseudo_preds[H] + w_gbt * gbt_preds[H]) / total
                 for H in HORIZONS}
        ens   = enforce_monotonicity(ens)

        # -- Distance rules --
        ens = apply_distance_rules(ens, dist_arr[val])

        rows.append(evaluate(ye[val], yt[val], ens, f'Full fold-{fold}'))
        print(f'  Fold {fold}: Hybrid={rows[-1]["Hybrid"]:.5f}  '
              f'C-index={rows[-1]["C-index"]:.4f}  W-Brier={rows[-1]["W-Brier"]:.4f}')

    df   = pd.DataFrame(rows)
    mean = df[['C-index', 'W-Brier', 'Hybrid']].mean()
    std  = df[['C-index', 'W-Brier', 'Hybrid']].std()
    return mean.to_dict(), std.to_dict(), df


print('Running full pipeline CV...')
full_mean, full_std, full_cv_df = cv_full_pipeline(
    X_train, y_event, y_time, dist_train,
    TUNED_PSEUDO_PARAMS, GBT_PARAMS,
    PSEUDO_WEIGHT, SURVIVAL_WEIGHT
)

print(f"\nFull pipeline CV:")
print(f"  Hybrid  = {full_mean['Hybrid']:.5f} ± {full_std['Hybrid']:.5f}")
print(f"  C-index = {full_mean['C-index']:.5f}")
print(f"  W-Brier = {full_mean['W-Brier']:.5f}")

In [ ]:
# ── Comprehensive comparison table ──
comparison = pd.DataFrame([
    {'Approach': 'GBT survival only',                 **{k: gbt_mean[k] for k in ['C-index','W-Brier','Hybrid']}},
    {'Approach': 'GBT + distance rules',              **{k: gbt_ruled_mean[k] for k in ['C-index','W-Brier','Hybrid']}},
    {'Approach': 'Pseudo-obs XGBoost only',           **{k: pseudo_mean[k] for k in ['C-index','W-Brier','Hybrid']}},
    {'Approach': 'Pseudo-obs + GBT + dist rules',     **{k: full_mean[k] for k in ['C-index','W-Brier','Hybrid']}},
]).set_index('Approach')

def green_shade(s):
    norm = (s - s.min()) / (s.max() - s.min() + 1e-9)
    return [f'background-color: rgba(46,204,113,{v:.2f})' for v in norm]

def red_shade(s):
    norm = 1 - (s - s.min()) / (s.max() - s.min() + 1e-9)
    return [f'background-color: rgba(46,204,113,{v:.2f})' for v in norm]

comparison.style \
    .apply(red_shade,   subset=['W-Brier']) \
    .apply(green_shade, subset=['C-index', 'Hybrid'])

In [ ]:
# ── Tune PSEUDO_WEIGHT vs SURVIVAL_WEIGHT on CV ──
print('Scanning ensemble weight ratios (pseudo-obs vs GBT)...')
weight_results = []

for pw in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    sw = 1.0 - pw
    # Use fold-level OOF from already-computed results
    # Quick re-run with different weights (pseudo and gbt CV dfs already available)
    # We re-blend OOF predictions that were saved in full_cv_df
    # For simplicity, do a 3-fold quick scan
    skf3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    scores = []
    for tr, val in skf3.split(X_train, y_event):
        fold_pseudo  = compute_pseudo_observations(y_event[tr], y_time[tr], HORIZONS)
        pm           = train_pseudo_xgb(X_train.iloc[tr], fold_pseudo, TUNED_PSEUDO_PARAMS)
        pp           = predict_pseudo_xgb(pm, X_train.iloc[val])

        gm = GradientBoostingSurvivalAnalysis(**GBT_PARAMS)
        gm.fit(X_train.iloc[tr].values, Surv.from_arrays(y_event[tr], y_time[tr]))
        fns = gm.predict_survival_function(X_train.iloc[val].values)
        S   = np.array([[_safe_eval(f, t) for t in HORIZONS] for f in fns])
        gp  = {H: 1.0 - S[:, i] for i, H in enumerate(HORIZONS)}

        ens = enforce_monotonicity({H: pw*pp[H] + sw*gp[H] for H in HORIZONS})
        ens = apply_distance_rules(ens, dist_train[val])
        scores.append(evaluate(y_event[val], y_time[val], ens)['Hybrid'])

    weight_results.append({'pseudo_weight': pw, 'gbt_weight': sw,
                           'CV Hybrid (3-fold)': np.mean(scores)})
    print(f'  pseudo={pw:.1f}, gbt={sw:.1f}: {np.mean(scores):.5f}')

weight_df = pd.DataFrame(weight_results)
best_row  = weight_df.loc[weight_df['CV Hybrid (3-fold)'].idxmax()]
print(f'\nBest weights: pseudo={best_row.pseudo_weight:.1f}, gbt={best_row.gbt_weight:.1f}')
print('Update PSEUDO_WEIGHT / SURVIVAL_WEIGHT in Section 0 with these values.')

## 9. Final Predictions & Submission

In [ ]:
# ── Retrain on full training set ──

# Pseudo-obs XGBoost
print('Training pseudo-obs XGBoost on full data...')
final_pseudo_models = train_pseudo_xgb(X_train, pseudo_obs, TUNED_PSEUDO_PARAMS)
test_pseudo_probs   = predict_pseudo_xgb(final_pseudo_models, X_test)

# GBT survival
print('Training GBT survival on full data...')
final_gbt = GradientBoostingSurvivalAnalysis(**GBT_PARAMS)
final_gbt.fit(X_train.values, Surv.from_arrays(y_event, y_time))
fns_test  = final_gbt.predict_survival_function(X_test.values)
S_test    = np.array([[_safe_eval(f, t) for t in HORIZONS] for f in fns_test])
test_gbt_probs = {H: 1.0 - S_test[:, i] for i, H in enumerate(HORIZONS)}

print('Both models trained.')

In [ ]:
# ── Build final predictions ──
total_w = PSEUDO_WEIGHT + SURVIVAL_WEIGHT
test_ensemble = {
    H: (PSEUDO_WEIGHT * test_pseudo_probs[H] + SURVIVAL_WEIGHT * test_gbt_probs[H]) / total_w
    for H in HORIZONS
}
test_ensemble = enforce_monotonicity(test_ensemble)
test_final    = apply_distance_rules(test_ensemble, dist_test)

# Verify monotonicity
for i in range(len(X_test)):
    p = [test_final[H][i] for H in HORIZONS]
    assert p == sorted(p), f'Monotonicity violated at row {i}'
print('Monotonicity check passed.')

print('\nTest prediction summary:')
for H in HORIZONS:
    print(f'  prob_{H}h: mean={test_final[H].mean():.4f}  '
          f'min={test_final[H].min():.4f}  max={test_final[H].max():.4f}')

# Show effect of distance rules on test
n_far   = (dist_test > FAR_DIST_M).sum()
n_close = (dist_test < CLOSE_DIST_M).sum()
print(f'\nDistance rules applied to: {n_close} close fires, {n_far} far fires')

In [ ]:
# ── Save submission ──
sub = pd.DataFrame({
    'event_id': test_raw['event_id'].values,
    'prob_12h': test_final[12],
    'prob_24h': test_final[24],
    'prob_48h': test_final[48],
    'prob_72h': test_final[72],
})
sample = pd.read_csv(DATA_DIR + 'sample_submission.csv')
assert list(sub.columns)    == list(sample.columns)
assert set(sub['event_id']) == set(sample['event_id'])

sub.to_csv('submission_v5.csv', index=False)
print('submission_v5.csv saved.')
print(sub.head(10))

In [ ]:
# ── Distribution plots ──
fig, axes = plt.subplots(2, 4, figsize=(18, 8))

for col, H in enumerate(HORIZONS):
    # Before distance rules
    axes[0, col].hist(test_ensemble[H], bins=20, color='#3498db',
                      edgecolor='white', alpha=0.8)
    axes[0, col].axvline(FAR_CAP, color='red', ls='--', lw=1,
                          label=f'FAR_CAP={FAR_CAP}')
    axes[0, col].set_title(f'prob_{H}h (before rules)')
    axes[0, col].legend(fontsize=8)

    # After distance rules
    axes[1, col].hist(test_final[H], bins=20, color='#e74c3c',
                      edgecolor='white', alpha=0.8)
    axes[1, col].set_title(f'prob_{H}h (after rules)')

plt.suptitle('Test predictions before vs after distance rules', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Summary ──
print('=' * 65)
print('  NOTEBOOK 05 SUMMARY')
print('=' * 65)
print(f'  CV scores (5-fold unbiased):')
print(f'  GBT survival only         : Hybrid={gbt_mean["Hybrid"]:.5f}')
print(f'  GBT + distance rules      : Hybrid={gbt_ruled_mean["Hybrid"]:.5f}')
print(f'  Pseudo-obs XGBoost only   : Hybrid={pseudo_mean["Hybrid"]:.5f}')
print(f'  Full pipeline             : Hybrid={full_mean["Hybrid"]:.5f}')
print()
print(f'  Pipeline: pseudo={PSEUDO_WEIGHT}, gbt={SURVIVAL_WEIGHT}')
print(f'  Distance rules: close<{CLOSE_DIST_M/1000:.0f}km floor={CLOSE_FLOOR}, '
      f'far>{FAR_DIST_M/1000:.0f}km cap={FAR_CAP}')
print(f'  Submission: submission_v5.csv')
print('=' * 65)